[Markdown Cell]
# Model B — audit your attached training output and generate its submission

Use this notebook for **any one of your Model A/B/C/D saved training runs**, or add
all four to `JOBS` and process them sequentially. It identifies the architecture,
reads the original saved code (or embedded notebook runtime), loads its checkpoint, runs sanity
checks, and exports a verified official-test submission ZIP. It never starts
training or executes the uploaded notebook's cells.

**Attach three inputs on Kaggle:** the saved training run, the verified Notebook 00
preprocessing export, and the official `Phase1- Test.json`. A Kaggle notebook input
normally exposes output files, including `model_b_run/code/model_b/` and
`model_b_run/checkpoints/`; the exported `.ipynb` is optional when saved code exists.
An `.ipynb` contains code/logs, **not trained weights**.
Checkpoint `.pt` files need their `.pt.json` SHA-256 receipts. The full saved run
is easiest to attach; notebook logs alone cannot reconstruct missing weights.
Use your original four notebooks; arbitrary unrelated model notebooks are unsupported.

Select a Kaggle GPU. Inference uses one GPU per job and releases it before the
next job. Both GPUs may be present, but no DDP training is needed here. All
reports, compact indices, caches, and submissions go under
`/kaggle/working/cs781_submissions/`. Input weights and token caches remain on the
read-only mount; the 21GB raw corpus and HF token are unnecessary.

Checks establish artifact integrity, completion, and numerical correctness.
They do not prove high-quality legal generation. Inspect development samples
and run the supplied held-out evaluator; private official references are available
only to Codabench. No upload to Codabench is performed by this notebook.

This copy selects Model B directly and includes the exact Torch version recorded in your saved Model B training notebook. Attach the training output, preprocessing export, and official test prompts, then run all cells.


[Markdown Cell]
## 1. Environment
Use Kaggle's CUDA-enabled PyTorch >=2.5. Install only the small tokenizer/progress packages. The helper runtime is embedded below; no companion workspace files are required.


In [1]:
# [Code Cell]
%pip install -q "tokenizers==0.22.1" "tqdm==4.67.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 37.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
transformers 5.16.1 requires tokenizers<0.24.0,>=0.23.1, but you have tokenizers 0.22.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# [Code Cell]
from pathlib import Path
from dataclasses import dataclass
import json
import os
import sys
from IPython.display import display, FileLink

module_parts: dict[str, list[str]] = {}

def append_module(filename: str, section: str) -> None:
    assert filename.endswith(".py") and "/" not in filename and "\\" not in filename
    module_parts.setdefault(filename, []).append(section)


[Markdown Cell]
## 2. Select your saved model run
For your attached Model B output, leave `SubmissionJob(model='b')`: the exact mounted run path is discovered from its saved code/config/checkpoints. No placeholder paths are needed. To use A/C/D, change the model letter or list more jobs. Set an exact `run_artifacts_dir` if several runs match. An optional `notebook_path` retains the original notebook-based workflow. `OFFICIAL_TEST_JSON=None` discovers one schema-checked Phase-1 test file. Without it, the model audit still runs and explains why no ZIP can be generated.


In [3]:
# [Code Cell]
@dataclass(frozen=True)
class SubmissionJob:
    notebook_path: str | None = None
    run_artifacts_dir: str | None = None
    model: str | None = None

JOBS = [
    SubmissionJob(model="b"),
    # SubmissionJob(model="a"),
    # SubmissionJob(model="c"),
    # SubmissionJob(model="d"),
    # Exact folder: SubmissionJob(run_artifacts_dir="/kaggle/input/.../model_b_run", model="b"),
    # Optional .ipynb: SubmissionJob(notebook_path="/kaggle/input/.../modelb-decoder.ipynb"),
]
OFFICIAL_TEST_JSON: str | None = None
PREPROCESSED_DATA_DIR = "/kaggle/input/cs781-preprocessed-data"
OUTPUT_ROOT = Path("/kaggle/working/cs781_submissions")
DEVICE = "cuda:0"
TRAINING_TORCH_VERSION: str | None = "2.10.0+cu128"  # Recorded in your saved Model B logs; fingerprint proof remains mandatory.

# A complete training run is required by default.
ALLOW_PARTIAL_TRAINING = False
GENERATION_TOKEN_LIMIT: int | None = None  # Keep the model's original bound.
VALIDATION_EXAMPLES_PER_SOURCE = 2        # 8 balanced reference examples.
GENERATE_DEVELOPMENT_SMOKE = True         # Inspect actual trained continuations.
GENERATE_OFFICIAL_SUBMISSIONS = True
MIN_FREE_BYTES = 1 << 30
MODEL_B_DECODING_RECOVERY = True
RUN_BERTSCORE_CHECK = True  # Needs Internet on first use; set False for the fastest ZIP-only run.
MODEL_B_RECOVERY_POLICY = {"version": "b-greedy-guarded-retries-v1", "repetition_penalty": 1.12,
    "no_repeat_ngram": 4, "max_stalled_tokens": 12, "temperature": 0.75, "top_p": 0.90,
    "prefer_trained_lengths": True}

settings = {
    "output_root": str(OUTPUT_ROOT), "official_test_json": OFFICIAL_TEST_JSON,
    "preprocessed_data_dir": PREPROCESSED_DATA_DIR, "input_root": "/kaggle/input",
    "search_roots": ["/kaggle/input", "/kaggle/working"], "device": DEVICE,
    "allow_partial_training": ALLOW_PARTIAL_TRAINING,
    "generation_token_limit": GENERATION_TOKEN_LIMIT,
    "validation_examples_per_source": VALIDATION_EXAMPLES_PER_SOURCE,
    "generate_development_smoke": GENERATE_DEVELOPMENT_SMOKE,
    "min_free_bytes": MIN_FREE_BYTES,
    "training_torch_version": TRAINING_TORCH_VERSION,
    "generation_policy": MODEL_B_RECOVERY_POLICY if MODEL_B_DECODING_RECOVERY else None,
}
assert JOBS and VALIDATION_EXAMPLES_PER_SOURCE >= 0 and MIN_FREE_BYTES >= 0
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


[Markdown Cell]
## 3. Reusable file helpers
Reject duplicate JSON keys and nonstandard values. Stream SHA-256 checks in 8MiB chunks. Publish reports and source atomically. The helper reads input files without moving or copying datasets/checkpoints.


In [4]:
# [Code Cell]
append_module('io_utils.py', r'''from pathlib import Path
import hashlib
import json
import os


def unique_object(pairs: list[tuple]) -> dict:
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key: {key!r}")
        result[key] = value
    return result


def parse_json(text: str):
    return json.loads(text, object_pairs_hook=unique_object,
                      parse_constant=lambda value: (_ for _ in ()).throw(ValueError(value)))


def read_json(path: Path, max_bytes: int = 256 << 20):
    if not path.is_file():
        raise FileNotFoundError(f"Missing file: {path}")
    if not 0 < path.stat().st_size <= max_bytes:
        raise ValueError(f"Empty or oversized JSON file: {path}")
    return parse_json(path.read_text(encoding="utf-8"))


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while block := stream.read(8 << 20):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    with temporary.open("w", encoding="utf-8", newline="\n") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write("\n"); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)


def atomic_source(path: Path, source: str) -> None:
    compile(source, str(path), "exec")
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    with temporary.open("w", encoding="utf-8", newline="\n") as stream:
        stream.write(source); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)
''')


[Markdown Cell]
## 4. Inspect the uploaded notebook
Parse notebook JSON and Python AST. Recover only literal `append_module` sections and the original package initializer. Require all eight runtime modules and identify A/B/C/D from its model class/version. Read execution counts, error outputs, completed-run summaries, and recorded Torch versions as evidence. Markdown is not executed; pip, original launch cells, and training cells are never run. The recovered model modules are imported later to run your own architecture.


In [5]:
# [Code Cell]
append_module('notebook.py', r'''from pathlib import Path
import ast
import hashlib
import json
import re
from .io_utils import read_json, sha256_file, atomic_json, atomic_source

MODEL_CLASSES = {"a": "LegalCausalDecoder", "b": "LegalEncoderDecoder",
                 "c": "LegalTransformerGRU", "d": "LegalSparseDecoder"}
MODULES = {"config.py", "model.py", "data.py", "state.py", "train.py", "infer.py", "sanity.py", "__init__.py"}


def identify_modules(modules: dict[str, str]) -> str:
    """Require a complete supported architecture before importing any runtime."""
    missing = MODULES - modules.keys()
    if missing:
        raise ValueError(f"Incomplete/unsupported embedded runtime; missing {sorted(missing)}. Use the original A/B/C/D notebook or its complete saved code folder.")
    for name, source in modules.items():
        compile(source, name, "exec")
    classes = {node.name for node in ast.parse(modules["model.py"]).body if isinstance(node, ast.ClassDef)}
    families = [family for family, name in MODEL_CLASSES.items() if name in classes]
    if len(families) != 1:
        raise ValueError("Could not identify exactly one supported Model A/B/C/D architecture")
    family = families[0]
    if f"cs781-model-{family}-v1" not in modules["config.py"]:
        raise ValueError("Model class and runtime version disagree")
    return family


def recover_notebook(path: Path) -> tuple[dict, dict[str, str]]:
    """Read only literal embedded sections; never run pip, training, or notebook cells.

    This adapter supports the four CS-781 notebooks prepared in this workspace.
    Recovered model modules are imported later to execute the user's own model.
    """
    notebook = read_json(path, 32 << 20)
    if notebook.get("nbformat") != 4 or not isinstance(notebook.get("cells"), list):
        raise ValueError("Expected a valid nbformat-4 .ipynb")
    parts, observations, errors, versions, executed = {}, [], [], set(), 0
    for index, cell in enumerate(notebook["cells"]):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        executed += cell.get("execution_count") is not None
        if "%pip" not in source:
            try:
                tree = ast.parse(source)
            except SyntaxError as exc:
                raise ValueError(f"Code cell {index + 1} has invalid Python: {exc.msg}") from exc
            for statement in tree.body:
                if isinstance(statement, ast.Expr) and isinstance(statement.value, ast.Call):
                    call = statement.value
                    if isinstance(call.func, ast.Name) and call.func.id == "append_module":
                        if len(call.args) != 2 or call.keywords:
                            raise ValueError("Unsupported dynamic append_module call")
                        name, body = map(ast.literal_eval, call.args)
                        if name not in MODULES or not isinstance(body, str):
                            raise ValueError(f"Unsupported runtime section: {name!r}")
                        parts.setdefault(name, []).append(body)
                if isinstance(statement, ast.Assign):
                    for target in statement.targets:
                        if (isinstance(target, ast.Subscript) and isinstance(target.value, ast.Name)
                                and target.value.id == "module_parts"
                                and isinstance(target.slice, ast.Constant)
                                and target.slice.value == "__init__.py"):
                            value = ast.literal_eval(statement.value)
                            if not isinstance(value, list) or not all(isinstance(item, str) for item in value):
                                raise ValueError("Unsupported initializer source")
                            parts["__init__.py"] = value
        for output in cell.get("outputs", []):
            if output.get("output_type") == "error":
                errors.append({"cell": index + 1, "type": output.get("ename"), "message": output.get("evalue")})
            text = "".join(output.get("text", []))
            versions.update(re.findall(r"PyTorch:\s*(\S+)", text))
            decoder, start = json.JSONDecoder(), text.find("{")
            while start >= 0:
                try:
                    value, end = decoder.raw_decode(text[start:])
                except ValueError:
                    start = text.find("{", start + 1); continue
                if isinstance(value, dict) and "status" in value and "history" in value:
                    observations.append(value)
                start = text.find("{", start + end)
    modules = {name: "".join(sections) for name, sections in parts.items()}
    family = identify_modules(modules)
    audit = {"notebook": str(path.resolve()), "notebook_sha256": sha256_file(path),
             "input_kind": "notebook",
             "model": family, "model_class": MODEL_CLASSES[family], "cells": len(notebook["cells"]),
             "executed_code_cells": executed, "saved_error_outputs": errors,
             "training_summaries": observations[-3:], "training_torch_versions": sorted(versions),
             "embedded_runtime": "complete", "weights_in_notebook": False}
    return audit, modules


''')


[Markdown Cell]
Alternatively, read the eight original `code/model_[a-d]/*.py` files exported by training. Discover a unique saved run using its `train_config.json`, committed checkpoint receipts, and requested model letter. Keep exact source bytes and hashes. Read `training_summary.json` as context; actual checkpoint history remains authoritative. Arbitrary first-match notebooks or mismatched model folders are never chosen.


In [6]:
# [Code Cell]
append_module('notebook.py', r'''def saved_run_candidates(roots: list[str], model: str | None = None) -> list[Path]:
    """Find complete code/checkpoint exports; never select an arbitrary .ipynb."""
    if model is not None and model not in MODEL_CLASSES:
        raise ValueError("model must be a, b, c, d, or None")
    candidates = set()
    for root_name in roots:
        root = Path(root_name)
        if not root.is_dir():
            continue
        for config in root.rglob("train_config.json"):
            run = config.parent.resolve()
            if not any((run / "checkpoints" / (name + ".json")).is_file()
                       for name in ("best.pt", "slot_0.pt", "slot_1.pt")):
                continue
            for family in MODEL_CLASSES:
                package = run / "code" / f"model_{family}"
                if (model is None or family == model) and all((package / name).is_file() for name in MODULES):
                    candidates.add(run)
    return sorted(candidates)


def recover_saved_run(run: Path) -> tuple[dict, dict[str, str]]:
    """Read the original exported Python files without executing training.

    The checkpoint fingerprint later authenticates these exact source hashes.
    Saved summaries provide context; the actual checkpoint cursor proves completion.
    """
    run = run.resolve()
    if run.name == "checkpoints":
        run = run.parent
    packages = [run / "code" / f"model_{family}" for family in MODEL_CLASSES
                if (run / "code" / f"model_{family}" / "config.py").is_file()]
    if len(packages) != 1:
        raise ValueError(f"Expected one saved runtime at {run}/code/model_[a-d]. Select the exact model run folder, or provide its original .ipynb.")
    package = packages[0]
    modules = {}
    for name in MODULES:
        path = package / name
        if not path.is_file():
            continue
        if not 0 < path.stat().st_size <= (4 << 20):
            raise ValueError(f"Empty or oversized runtime source: {path}")
        # Preserve original newline bytes used by the checkpoint source fingerprint.
        modules[name] = path.read_bytes().decode("utf-8")
    family = identify_modules(modules)
    if package.name != f"model_{family}":
        raise ValueError("Saved code folder and detected model disagree")
    if {path.name for path in package.glob("*.py")} != MODULES:
        raise ValueError("Unexpected extra Python source in saved runtime; use the exact original training code folder")
    summaries = []
    summary_path = run / "training_summary.json"
    if summary_path.is_file():
        summary = read_json(summary_path, 4 << 20)
        if not isinstance(summary, dict) or "status" not in summary or "history" not in summary:
            raise ValueError(f"Malformed training summary: {summary_path}")
        summaries.append(summary)
    identity = {name: hashlib.sha256(source.encode("utf-8")).hexdigest()
                for name, source in sorted(modules.items())}
    digest = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    audit = {"notebook": str(run), "notebook_sha256": digest, "input_kind": "saved_run",
             "model": family, "model_class": MODEL_CLASSES[family], "cells": 0,
             "executed_code_cells": 0, "saved_error_outputs": [],
             "training_summaries": summaries, "training_torch_versions": [],
             "embedded_runtime": "complete", "weights_in_notebook": False,
             "saved_code_dir": str(package), "source_hashes": identity}
    return audit, modules


''')


[Markdown Cell]
Write the recovered source byte-for-byte under an isolated per-notebook code directory. This preserves the source hashes used by the original checkpoint fingerprint. Each model gets its own reports and prediction cache. No A/B/C/D runtime is bundled as a replacement for your trained notebook.


In [7]:
# [Code Cell]
append_module('notebook.py', r'''def stage_job(notebook_path: str | None, run_artifacts_dir: str | None,
              settings: dict, model: str | None = None) -> Path:
    if notebook_path:
        audit, modules = recover_notebook(Path(notebook_path))
    else:
        if not run_artifacts_dir:
            candidates = saved_run_candidates(settings["search_roots"], model)
            if not candidates:
                raise FileNotFoundError("No complete saved model run was found. Attach your training notebook's output with code/model_[a-d], checkpoints/*.pt and .pt.json receipts, and train_config.json. Kaggle notebook inputs usually contain output files, not the .ipynb itself.")
            if len(candidates) != 1:
                raise ValueError("Several saved runs match. Set run_artifacts_dir to one exact model run folder:\n" + "\n".join(map(str, candidates)))
            run_artifacts_dir = str(candidates[0])
        audit, modules = recover_saved_run(Path(run_artifacts_dir))
    if model is not None and audit["model"] != model:
        raise ValueError(f"Selected model {model.upper()} but input contains model {audit['model'].upper()}")
    version = settings.get("training_torch_version")
    if version:
        if not isinstance(version, str) or re.fullmatch(r"\d+\.\d+\.\d+(?:\+[\w.]+)?", version) is None:
            raise ValueError("training_torch_version must be an exact version string, e.g. 2.10.0+cu128")
        audit["training_torch_versions"] = list(dict.fromkeys([*audit["training_torch_versions"], version]))
    job_dir = Path(settings["output_root"]) / f"model_{audit['model']}_{audit['notebook_sha256'][:12]}"
    package = job_dir / "code" / f"model_{audit['model']}"
    for name, source in modules.items():
        atomic_source(package / name, source)
    atomic_json(job_dir / "notebook_audit.json", audit)
    request = dict(settings, job_dir=str(job_dir.resolve()), notebook_audit=audit,
                   runtime_code_dir=str(package.parent.resolve()), run_artifacts_dir=run_artifacts_dir)
    request_path = job_dir / "request.json"
    atomic_json(request_path, request)
    return request_path
''')


[Markdown Cell]
## 5. Find the actual trained weights
Find committed `best.pt`/`slot_0.pt`/`slot_1.pt` receipts under an explicit saved-run folder, or under attached Kaggle inputs/current working files. Prefer a matching fingerprint from saved notebook outputs. Missing weights block generation; multiple matching runs require an exact folder path. A notebook with stripped outputs can still pass if its checkpoint proves compatibility and completion.


In [8]:
# [Code Cell]
append_module('artifacts.py', r'''from pathlib import Path
import math
from .io_utils import read_json, sha256_file

CHECKPOINT_NAMES = ("best.pt", "slot_0.pt", "slot_1.pt")


def checkpoint_directories(request: dict) -> list[Path]:
    explicit = request.get("run_artifacts_dir")
    roots = [Path(explicit)] if explicit else [Path(item) for item in request["search_roots"]]
    candidates = set()
    for root in roots:
        if not root.is_dir():
            if explicit:
                raise FileNotFoundError(f"Saved run folder does not exist: {root}")
            continue
        for name in CHECKPOINT_NAMES:
            for receipt in root.rglob(name + ".json"):
                candidates.add(receipt.parent.resolve())
    expected = {item["fingerprint"] for item in request["notebook_audit"]["training_summaries"]
                if isinstance(item.get("fingerprint"), str)}
    filtered = []
    for directory in sorted(candidates):
        fingerprints = set()
        for name in CHECKPOINT_NAMES:
            try:
                fingerprints.add(read_json(directory / (name + ".json"), 4 << 20)["fingerprint"])
            except (OSError, ValueError, KeyError):
                pass
        if expected and not expected.intersection(fingerprints):
            continue
        config_path = directory.parent / "train_config.json"
        if not expected and config_path.is_file():
            config = read_json(config_path, 4 << 20)
            family = "b" if "encoder_layers" in config else "c" if "gru_layers" in config else "d" if "local_window" in config else "a"
            if family != request["notebook_audit"]["model"]:
                continue
        filtered.append(directory)
    if not filtered:
        raise FileNotFoundError("No matching full checkpoint + .pt.json receipt was found. Attach the saved training run as Kaggle input; an .ipynb alone has no weights.")
    if len(filtered) > 1:
        raise ValueError("Several matching saved runs exist. Set run_artifacts_dir to one exact checkpoints folder:\n" + "\n".join(map(str, filtered)))
    return filtered


''')


[Markdown Cell]
Verify receipt fields, byte sizes, SHA-256, and file stability. Ignore a corrupt/incomplete slot if another intact checkpoint exists. Reject mixed experiment fingerprints. Use the newest committed checkpoint to establish training completion, then the verified best checkpoint for inference (latest if no best exists). A best checkpoint from an earlier epoch does not mean the overall training run is incomplete.


In [9]:
# [Code Cell]
append_module('artifacts.py', r'''def verified_checkpoints(directory: Path) -> tuple[list[dict], list[str]]:
    valid, problems = [], []
    for name in CHECKPOINT_NAMES:
        path, receipt_path = directory / name, directory / (name + ".json")
        if not path.exists() and not receipt_path.exists():
            continue
        try:
            receipt = read_json(receipt_path, 4 << 20)
            if not all(type(receipt.get(key)) is int for key in ("bytes", "step", "epoch", "generation")):
                raise ValueError("Invalid integer receipt fields")
            if receipt["bytes"] <= 0 or min(receipt["step"], receipt["epoch"]) < 0 or receipt["generation"] < 1:
                raise ValueError("Invalid receipt cursor")
            if not isinstance(receipt.get("fingerprint"), str) or not isinstance(receipt.get("sha256"), str):
                raise ValueError("Missing checkpoint identity")
            score = receipt.get("validation_nll")
            if score is not None and (type(score) not in (int, float) or not math.isfinite(score) or score < 0):
                raise ValueError("Invalid held-out score")
            before = path.stat()
            if before.st_size != receipt["bytes"] or sha256_file(path) != receipt["sha256"]:
                raise ValueError("Checkpoint size/SHA-256 mismatch")
            after = path.stat()
            if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns) or read_json(receipt_path) != receipt:
                raise ValueError("Checkpoint changed during verification; stop training before exporting")
            valid.append({"path": str(path), "receipt": receipt, "mtime_ns": after.st_mtime_ns})
        except (OSError, ValueError, TypeError, KeyError) as exc:
            problems.append(f"{name}: {type(exc).__name__}: {exc}")
    if not valid:
        raise ValueError(f"No intact committed checkpoint in {directory}: {problems}")
    fingerprints = {item["receipt"]["fingerprint"] for item in valid}
    if len(fingerprints) != 1:
        raise ValueError("Mixed experiments in one checkpoint folder; separate the saved runs")
    return valid, problems


def choose_checkpoints(valid: list[dict]) -> tuple[dict, dict]:
    latest = max(valid, key=lambda item: (item["receipt"]["step"], item["receipt"]["generation"]))
    best = [item for item in valid if Path(item["path"]).name == "best.pt"]
    return latest, best[0] if best else latest


''')


[Markdown Cell]
## 6. Official input and output contract
Require each exact string ID to have a 512-word prompt. Final records must contain only `text`, with exactly 128 continuation words, no control tokens, and no copied 128-word prompt prefix/tail. All official IDs must be covered; extra IDs are rejected. Word count uses whitespace splitting, consistent with the course notebook convention.


In [10]:
# [Code Cell]
append_module('artifacts.py', r'''def official_prompts(path: Path) -> dict:
    records = read_json(path)
    if not isinstance(records, dict) or not records:
        raise ValueError("Official test JSON must map IDs to prompt records")
    for identifier, row in records.items():
        if not isinstance(identifier, str) or not identifier.strip() or not isinstance(row, dict):
            raise ValueError(f"Invalid official record: {identifier!r}")
        if not isinstance(row.get("text"), str) or len(row["text"].split()) != 512:
            raise ValueError(f"Official ID {identifier!r} does not contain a 512-word prompt. Use Phase1- Test.json, not development predictions.")
    return records


def validate_predictions(predictions: dict, prompts: dict) -> dict:
    if not isinstance(predictions, dict) or set(predictions) != set(prompts):
        raise ValueError("Prediction IDs differ from the official test IDs")
    for identifier, row in predictions.items():
        if not isinstance(row, dict) or set(row) != {"text"} or not isinstance(row["text"], str):
            raise ValueError(f"Invalid prediction record: {identifier}")
        words = row["text"].split()
        if len(words) != 128:
            raise ValueError(f"ID {identifier}: expected 128 words, found {len(words)}")
        if any(token in row["text"] for token in ("<pad>", "<s>", "</s>", "<unk>", "<sep>", "<mask>")):
            raise ValueError(f"Control token in prediction {identifier}")
        prompt_words = prompts[identifier]["text"].split()
        if words == prompt_words[:128] or words == prompt_words[-128:]:
            raise ValueError(f"Prediction {identifier} copies a prompt segment; inspect model output")
    return {"examples": len(predictions), "exact_ids": True, "words_per_continuation": 128}
''')


[Markdown Cell]
Discover a uniquely named `Phase1- Test.json` or `test.json` and validate every prompt; ignore development prediction/truth and preprocessing JSON files. Several valid copies require an explicit path. A missing official test file does not prevent trained-model auditing. Official ZIP export requires the real test prompts, and the final cell explains missing inputs clearly.


In [11]:
# [Code Cell]
append_module('inputs.py', r'''from pathlib import Path
import re
from .artifacts import official_prompts


def resolve_official_test(request: dict, required: bool) -> Path | None:
    """Accept an explicit test file or one uniquely named, schema-checked input.

    Development predictions/truth/tokenizer files never qualify by filename.
    Auditing a model may proceed without official prompts; export requires them.
    """
    explicit = request.get("official_test_json")
    if explicit:
        path = Path(explicit)
        official_prompts(path)
        return path.resolve()
    candidates, problems = set(), []
    allowed_names = {"phase1test", "testphase1", "test", "testprompts", "phase1testprompts", "phase1publictest"}
    for root_name in request["search_roots"]:
        root = Path(root_name)
        if not root.is_dir():
            continue
        for path in root.rglob("*.json"):
            if re.sub(r"[^a-z0-9]", "", path.stem.lower()) not in allowed_names:
                continue
            try:
                official_prompts(path)
                candidates.add(path.resolve())
            except (OSError, ValueError, TypeError, KeyError) as error:
                problems.append(f"{path}: {error}")
    if len(candidates) > 1:
        raise ValueError("Several valid official test files were found. Set OFFICIAL_TEST_JSON to one exact path:\n" + "\n".join(map(str, sorted(candidates))))
    if candidates:
        return next(iter(candidates))
    if required:
        detail = "\nRejected named candidates:\n" + "\n".join(problems[:5]) if problems else ""
        raise FileNotFoundError("Official Phase-1 test prompts are missing. Attach Phase1- Test.json or set OFFICIAL_TEST_JSON to its exact mounted path. The model can be audited without it, but a submission ZIP requires the official 512-word prompts." + detail)
    return None
''')


[Markdown Cell]
## 7. Check checkpoint completeness
Load with `torch.load(weights_only=True, mmap=True, map_location='cpu')`. Verify model, initialized optimizer, scaler, scheduler, epoch/step/cursor, and two rank RNG states. Require successful updates and contiguous finite held-out epoch history. `epoch == planned epochs` with `next_batch == 0` establishes completion. Default policy blocks a partial run. This utility does not restore an optimizer or resume training. [PyTorch checkpoint loading](https://docs.pytorch.org/docs/stable/generated/torch.load.html).


In [12]:
# [Code Cell]
append_module('checks.py', r'''from dataclasses import asdict, replace
from pathlib import Path
import gc
import hashlib
import json
import math
import torch
from torch.nn import functional as F
from .io_utils import sha256_file

FINGERPRINT_PATH_FIELDS = ("data_dir", "run_dir", "resume_dir", "max_run_seconds", "min_free_bytes",
                           "max_checkpoint_bytes", "generation_examples_per_source", "generation_token_limit")


def load_payload(candidate: dict) -> dict:
    """Map tensor storage read-only; never copy optimizer moments into working files."""
    path, receipt = Path(candidate["path"]), candidate["receipt"]
    before = path.stat()
    if before.st_size != receipt["bytes"] or before.st_mtime_ns != candidate["mtime_ns"]:
        raise ValueError("Checkpoint changed after its hash verification")
    payload = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
    required = {"version", "fingerprint", "world_size", "config", "epoch", "step", "model",
                "optimizer", "scaler", "scheduler", "cursor", "rng_by_rank"}
    if not isinstance(payload, dict) or required - payload.keys():
        raise ValueError("Incomplete training checkpoint; expected model/optimizer/scaler/scheduler/cursor/RNG state")
    if payload["fingerprint"] != receipt["fingerprint"] or payload["step"] != receipt["step"] or payload["epoch"] != receipt["epoch"]:
        raise ValueError("Checkpoint payload and receipt disagree")
    if type(payload["step"]) is not int or type(payload["epoch"]) is not int or payload["world_size"] != 2:
        raise ValueError("Invalid saved epoch/step/world size")
    cursor = payload["cursor"]
    if not isinstance(cursor, dict) or not all(type(cursor.get(key)) is int for key in ("epoch", "next_batch", "global_step", "optimizer_steps")):
        raise ValueError("Missing exact training cursor")
    if cursor["global_step"] != payload["step"] or cursor["epoch"] != payload["epoch"] or min(cursor["epoch"], cursor["next_batch"]) < 0:
        raise ValueError("Saved cursor disagrees with epoch/step")
    if not 0 < cursor["optimizer_steps"] <= cursor["global_step"]:
        raise ValueError("No successful optimizer updates are recorded")
    if not isinstance(payload["optimizer"], dict) or not payload["optimizer"].get("state") or not payload["optimizer"].get("param_groups"):
        raise ValueError("Missing initialized optimizer state")
    if not isinstance(payload["scaler"], dict) or not isinstance(payload["scheduler"], dict):
        raise ValueError("Missing scaler/scheduler state")
    if payload["scheduler"].get("steps") != cursor["optimizer_steps"] or len(payload["rng_by_rank"]) != 2:
        raise ValueError("Scheduler/RNG state does not match the training cursor")
    after = path.stat()
    if (after.st_size, after.st_mtime_ns) != (before.st_size, before.st_mtime_ns):
        raise ValueError("Checkpoint changed while loading")
    return payload


def training_evidence(payload: dict) -> dict:
    cursor, cfg = payload["cursor"], payload["config"]
    complete = cursor["epoch"] == cfg["epochs"] and cursor["next_batch"] == 0
    if cursor["epoch"] > cfg["epochs"]:
        raise ValueError("Saved epoch exceeds the configured training plan")
    history = cursor.get("history", [])
    if not isinstance(history, list) or len(history) != cursor["epoch"]:
        raise ValueError("Missing completed-epoch validation history")
    for index, row in enumerate(history, start=1):
        if row.get("epoch") != index:
            raise ValueError("Validation history epochs are not contiguous")
        for key in ("train_nll", "conditional_nll"):
            value = row.get(key)
            if type(value) not in (int, float) or not math.isfinite(value) or value < 0:
                raise ValueError(f"Invalid {key} in epoch {index}")
        if row.get("target_tokens", 0) <= 0 or row.get("examples", 0) <= 0:
            raise ValueError("Validation has no scored target tokens")
    indicators = {"scope": "Saved learning trends; generation quality requires inspecting/scoring actual continuations",
                  "skipped_amp_updates": cursor["global_step"] - cursor["optimizer_steps"],
                  "skipped_update_fraction": (cursor["global_step"] - cursor["optimizer_steps"]) / cursor["global_step"]}
    if history:
        first, last = history[0], history[-1]
        indicators.update(initial_validation_nll=first["conditional_nll"],
            final_validation_nll=last["conditional_nll"],
            validation_nll_improved=last["conditional_nll"] < first["conditional_nll"],
            validation_improved_each_epoch=all(new["conditional_nll"] <= old["conditional_nll"]
                                              for old, new in zip(history, history[1:])),
            initial_validation_perplexity=math.exp(min(first["conditional_nll"], 80)),
            final_validation_perplexity=math.exp(min(last["conditional_nll"], 80)),
            final_train_validation_nll_gap=last["conditional_nll"] - last["train_nll"],
            best_validation_epoch=min(history, key=lambda row: row["conditional_nll"])["epoch"])
    return {"complete": complete, "completed_epochs": cursor["epoch"], "planned_epochs": cfg["epochs"],
            "next_batch": cursor["next_batch"], "saved_step": cursor["global_step"],
            "successful_optimizer_updates": cursor["optimizer_steps"], "history": history,
            "learning_indicators": indicators}


''')


[Markdown Cell]
## 8. Compatibility and parameter boundary
Reproduce the saved fingerprint using exact recovered source, checkpoint learning
configuration, verified tokenizer/data receipts, and the recorded training Torch
version (or an explicit exact version hint). A hint is accepted only if it
reproduces the full saved fingerprint. Input/output paths and generation token limit may change; architecture
and training settings may not. Inference across Torch versions is reported only
when the original fingerprint can be reproduced; missing evidence blocks loading.

Construct the architecture on `meta`, strictly assign checkpoint tensors, count
unique trainable Parameters, and require
\[50{,}000{,}000<P<300{,}000{,}000.\]
Scan saved FP32 weights for nonfinite values, then move only model weights to the
GPU. Optimizer moments remain memory-mapped on input and are not copied to working
storage. [Meta + assign loading](https://docs.pytorch.org/tutorials/recipes/recipes/module_load_state_dict_tips.html).


In [13]:
# [Code Cell]
append_module('checks.py', r'''def experiment_choices(config: dict) -> dict:
    return {key: value for key, value in config.items() if key not in FINGERPRINT_PATH_FIELDS}


def prove_fingerprint(cfg, contract: dict, runtime, request: dict, expected: str) -> str:
    package = Path(runtime["state"].__file__).parent
    sources = {path.name: sha256_file(path) for path in sorted(package.glob("*.py"))}
    versions = list(dict.fromkeys([str(torch.__version__), *request["notebook_audit"]["training_torch_versions"]]))
    for version in versions:
        value = {"version": runtime["config"].RUNTIME_VERSION, "config": experiment_choices(asdict(cfg)),
                 "data": contract["fingerprint"], "sources": sources, "torch": version}
        actual = hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()
        if actual == expected:
            return version
    raise ValueError("Checkpoint fingerprint does not match notebook source + saved configuration + tokenizer/data + recorded PyTorch version. Use its original trained notebook and exact preprocessing export. A stripped notebook from a different Torch version may lack the version evidence needed here.")


def parameter_boundary(model) -> int:
    count = sum(parameter.numel() for parameter in {id(p): p for p in model.parameters() if p.requires_grad}.values())
    if not 50_000_000 < count < 300_000_000:
        raise ValueError(f"Unique trainable parameter boundary violated: {count:,}")
    return count


def load_trained_model(payload: dict, cfg, runtime: dict, model_class: str, device: torch.device) -> tuple[torch.nn.Module, dict]:
    """Construct on meta, assign verified FP32 checkpoint tensors, then transfer weights."""
    constructor = getattr(runtime["model"], model_class)
    with torch.device("meta"):
        model = constructor(cfg)
    count = parameter_boundary(model)
    model.load_state_dict(payload["model"], strict=True, assign=True)
    if parameter_boundary(model) != count:
        raise ValueError("Parameter sharing changed when checkpoint weights were assigned")
    checked = 0
    for name, tensor in model.state_dict().items():
        if not tensor.is_floating_point():
            continue
        if tensor.dtype != torch.float32:
            raise ValueError(f"Expected FP32 saved master weights: {name} is {tensor.dtype}")
        flat = tensor.reshape(-1)
        for start in range(0, flat.numel(), 1_000_000):
            if not bool(torch.isfinite(flat[start:start + 1_000_000]).all()):
                raise ValueError(f"Non-finite trained weights: {name}")
        checked += tensor.numel()
    model.to(device).eval()
    return model, {"unique_trainable_parameters": count, "strict_state_dict": "passed",
                   "finite_fp32_weight_values_checked": checked, "checkpoint_weights_loaded": True}


''')


[Markdown Cell]
## 9. Test the loaded trained model
Use the actual checkpoint weights, eval mode, and FP32 probes. Check finite shapes, decoder causality, padding isolation, and cached/full logits. B uses self/cross K/V; C checks FP32 recurrent state; D crosses its real window/landmark boundaries. These tests do not initialize a second random model or perform backward/optimizer steps.


In [14]:
# [Code Cell]
append_module('checks.py', r'''@torch.inference_mode()
def trained_forward_checks(model, family: str, runtime: dict, device: torch.device) -> dict:
    """Check finite logits, causal isolation, padding, and full/cached agreement."""
    vocab = model.cfg.vocab_size
    ids = torch.randint(6, vocab, (2, 12), device=device)
    mask = torch.ones_like(ids, dtype=torch.bool); mask[1, -3:] = False
    changed = ids.clone(); changed[:, 6:] = (changed[:, 6:] + 11 - 6) % (vocab - 6) + 6
    padded = ids.clone(); padded[1, -3:] = 9
    if family == "b":
        source, source_mask = ids.clone(), mask
        decoder = torch.randint(6, vocab, (2, 12), device=device)
        decoder_mask = torch.ones_like(decoder, dtype=torch.bool)
        logits = model(source, source_mask, decoder, decoder_mask)
        altered = decoder.clone(); altered[:, 6:] = (altered[:, 6:] + 11 - 6) % (vocab - 6) + 6
        torch.testing.assert_close(logits[:, :6], model(source, source_mask, altered, decoder_mask)[:, :6], rtol=2e-4, atol=2e-4)
        torch.testing.assert_close(logits, model(padded, source_mask, decoder, decoder_mask), rtol=2e-4, atol=2e-4)
        memory = model.encode(source[:1], source_mask[:1])
        cache = runtime["model"].prepare_decoder_cache(model, memory)
        cached = torch.stack([runtime["model"].decode_one_token(model, decoder[:1, position:position+1],
                            source_mask[:1], cache) for position in range(12)], dim=1)
        full = logits[:1]
        length = 12
    else:
        logits = model(ids, mask)
        torch.testing.assert_close(logits[:, :6], model(changed, mask)[:, :6], rtol=2e-4, atol=2e-4)
        torch.testing.assert_close(logits[1, :9], model(padded, mask)[1, :9], rtol=2e-4, atol=2e-4)
        length = min(model.cfg.max_seq_tokens, max(model.cfg.local_window + 9, model.cfg.global_stride + 17)) if family == "d" else 12
        sequence = torch.randint(6, vocab, (1, length), device=device)
        prefix = max(1, length - 20) if family == "d" else 4
        full = model(sequence)[:, prefix - 1:]
        first, cache = model.prefill(sequence[:, :prefix])
        values = [first]
        for position in range(prefix, length):
            values.append(runtime["model"].decode_one_token(model, sequence[:, position:position+1], cache))
        cached = torch.stack(values, dim=1)
        if family == "c" and not all(state.dtype == torch.float32 for state in cache.recurrent_states):
            raise ValueError("Model C recurrent states lost FP32 precision")
    if logits.shape != (2, 12, vocab) or not bool(torch.isfinite(logits).all()) or not bool(torch.isfinite(cached).all()):
        raise ValueError("Invalid/non-finite trained-model logits")
    torch.testing.assert_close(full, cached, rtol=2e-4, atol=2e-4)
    return {"finite_logits": True, "logit_shape": list(logits.shape), "causal_isolation": "passed",
            "padding_isolation": "passed", "trained_cached_decode": "passed", "cache_test_length": length,
            "cached_logits_max_abs_error": float((full - cached).abs().max()), "probe_precision": "FP32"}


''')


[Markdown Cell]
## 10. Small held-out learning check
Reuse the original tokenizer and teacher-forcing pipeline. Score only continuation
labels, including BOS/EOS conventions and documented BPE crops:
\[\mathrm{NLL}=-\frac{\sum \log p_\theta(y_t\mid\text{prompt},y_{<t})}{N_{\rm reference}}.\]
Select two examples per source by default. Report token-weighted NLL, perplexity,
and truncations; compare to uniform-vocabulary NLL only as a diagnostic. A small
sample is not the full validation set, and finite NLL alone cannot prove quality.
The original index verifier also checks whole-document train/validation isolation.


In [15]:
# [Code Cell]
append_module('checks.py', r'''def balanced_validation_records(cfg, runtime: dict, per_source: int) -> list[dict]:
    counts = dict.fromkeys(runtime["config"].SOURCES, 0)
    records = []
    with (Path(cfg.data_dir) / "validation_pairs.jsonl").open(encoding="utf-8") as stream:
        for line in stream:
            row = json.loads(line)
            source = row["source"]
            if source not in counts:
                raise ValueError(f"Unknown held-out source: {source}")
            if counts[source] < per_source:
                if len(row["prompt"].split()) != 512 or len(row["reference"].split()) != 128:
                    raise ValueError("Malformed held-out prompt/reference lengths")
                counts[source] += 1; records.append(row)
            if all(count == per_source for count in counts.values()):
                break
    if not all(count == per_source for count in counts.values()):
        raise ValueError(f"Not enough balanced held-out records: {counts}")
    return records


@torch.inference_mode()
def heldout_loss(model, cfg, family: str, runtime: dict, records: list[dict], device: torch.device) -> dict:
    total, tokens, prompt_crops, target_crops = 0.0, 0, 0, 0
    for row in records:
        prefix, target, p_crop, t_crop = runtime["data"].tokenize_pair(
            runtime["tokenizer"], row["prompt"], row["reference"], cfg)
        prompt_crops += p_crop; target_crops += t_crop
        item = {"source": prefix, "target": target} if family == "b" else {
            "input_ids": (prefix + target)[:-1], "labels": [-100] * (len(prefix) - 1) + target}
        batch = runtime["train"].move_batch(runtime["data"].collate_continuations([item]), device)
        with runtime["train"].autocast_context(device):
            loss = runtime["train"].summed_loss(model, batch)
        if not bool(torch.isfinite(loss)):
            raise ValueError("Non-finite held-out reference loss")
        total += float(loss); tokens += int(batch["labels"].ne(-100).sum())
    if tokens <= 0:
        raise ValueError("Held-out sanity check contains no supervised reference tokens")
    nll = total / tokens
    return {"examples": len(records), "target_tokens": tokens, "conditional_nll": nll,
            "conditional_perplexity": math.exp(min(nll, 80)), "uniform_vocab_nll": math.log(cfg.vocab_size),
            "prompt_truncations": prompt_crops, "target_truncations": target_crops,
            "scope": "Small balanced held-out smoke sample; not the entire validation set or official private score"}
''')


[Markdown Cell]
Model B first keeps its original greedy decoding. If a prompt cannot close 128 words within its 1,024-position decoder, retry with a mild sign-aware repetition penalty and no repeated four-token BPE n-gram. Break stalled word fragments using model-ranked lexical boundary tokens, reserving positions to finish the continuation. A second attempt uses deterministic nucleus sampling. All attempts stay inside the trained architecture's positional capacity. This prevents a single stalled prompt from aborting generation; it does not guarantee high BERTScore or legal accuracy.


In [16]:
# [Code Cell]
append_module('decoding.py', r'''from pathlib import Path
import hashlib
import json
import torch

RECOVERY_POLICY = {"version": "b-greedy-guarded-retries-v1", "repetition_penalty": 1.12,
                   "no_repeat_ngram": 4, "max_stalled_tokens": 12,
                   "temperature": 0.75, "top_p": 0.90, "prefer_trained_lengths": True}


def lexical_boundary_ids(tokenizer) -> list[int]:
    """Find tokens that start a whitespace-separated lexical word, without filler.

    Byte-level tokens are tested through the real tokenizer decoder. Only the
    late/stalled retry guard uses this set; ordinary decoding keeps its vocabulary.
    """
    result = []
    for identifier in range(6, tokenizer.get_vocab_size()):
        text = tokenizer.decode([identifier], skip_special_tokens=True)
        stripped = text.lstrip()
        if text and text[0].isspace() and stripped and stripped[0].isalnum() and "\ufffd" not in stripped:
            result.append(identifier)
    return result


@torch.inference_mode()
def guarded_model_b(model, runtime: dict, prompt: str, device, token_limit: int,
                    policy: dict, sample: bool, seed: int) -> tuple[str, dict]:
    """Reuse trained source/self/cross K/V; keep every decoder position in bounds.

    Repetition penalty and no-repeat BPE n-grams discourage token loops. A lexical
    boundary guard reserves enough positions to close 128 words, without copying
    a reference/prompt or appending filler. Boundaries still use model-ranked tokens.
    """
    tokenizer = runtime["tokenizer"]
    model.eval()
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    source_limit = model.cfg.max_source_tokens - 1
    if policy.get("prefer_trained_lengths", True):
        source_limit = min(source_limit, getattr(model.cfg, "train_prefix_tokens", source_limit))
    source = torch.tensor([[1] + prompt_ids[-source_limit:]], device=device)
    source_mask = torch.ones_like(source, dtype=torch.bool)
    with runtime["train"].autocast_context(device):
        memory = model.encode(source, source_mask)
        cache = runtime["model"].prepare_decoder_cache(model, memory)
    limit = min(token_limit, model.cfg.max_decoder_tokens)
    if policy.get("prefer_trained_lengths", True):
        # Source/target positions beyond the training windows were not supervised.
        # The retry uses those seen in training, while retaining space for 129 words.
        trained_limit = max(129, getattr(model.cfg, "train_target_tokens", limit))
        limit = min(limit, trained_limit)
    token = torch.tensor([[1]], device=device)
    seen = torch.zeros(model.cfg.vocab_size, device=device, dtype=torch.bool)
    word_ids = runtime.get("lexical_boundary_ids")
    if word_ids is None:
        word_ids = lexical_boundary_ids(tokenizer)
        runtime["lexical_boundary_ids"] = word_ids
    boundary = torch.zeros_like(seen)
    if word_ids:
        boundary[torch.tensor(word_ids, device=device)] = True
    generator = torch.Generator(device=device).manual_seed(seed)
    generated, ngrams = [], {}
    words, previous_words, stalled, forced = [], 0, 0, 0
    n = int(policy["no_repeat_ngram"])
    penalty = float(policy["repetition_penalty"])
    for step in range(limit):
        with runtime["train"].autocast_context(device):
            logits = runtime["model"].decode_one_token(model, token, source_mask, cache).float()[0]
        if not bool(torch.isfinite(logits).all()):
            raise FloatingPointError("Non-finite Model B retry logits")
        logits = torch.where(seen, torch.where(logits < 0, logits * penalty, logits / penalty), logits)
        logits[:6] = -torch.inf
        if n > 0 and len(generated) >= n - 1:
            forbidden = ngrams.get(tuple(generated[-(n - 1):]), set())
            if forbidden:
                logits[list(forbidden)] = -torch.inf
        # Normal words require few BPE pieces. Break a stalled word or reserve the
        # final positions for lexical boundaries, chosen by the actual model.
        remaining = limit - step
        required = max(0, 129 - len(words))
        force_boundary = stalled >= int(policy["max_stalled_tokens"]) or remaining <= required + 4
        if force_boundary and word_ids:
            logits.masked_fill_(~boundary, -torch.inf)
            forced += 1
        if not bool(torch.isfinite(logits).any()):
            raise RuntimeError("Retry constraints removed every lexical candidate; inspect tokenizer/model output")
        if sample:
            sorted_logits, indices = torch.sort(logits / float(policy["temperature"]), descending=True)
            probabilities = torch.softmax(sorted_logits, dim=-1)
            remove = probabilities.cumsum(dim=-1) - probabilities >= float(policy["top_p"])
            probabilities.masked_fill_(remove, 0)
            probabilities /= probabilities.sum()
            chosen = int(indices[torch.multinomial(probabilities, 1, generator=generator)].item())
        else:
            chosen = int(logits.argmax().item())
        generated.append(chosen); seen[chosen] = True
        if n > 0 and len(generated) >= n:
            ngrams.setdefault(tuple(generated[-n:-1]), set()).add(chosen)
        token = torch.tensor([[chosen]], device=device)
        text = tokenizer.decode(generated, skip_special_tokens=True)
        words = text.split()
        stalled = stalled + 1 if len(words) <= previous_words else 0
        previous_words = len(words)
        if len(words) >= 129:
            return " ".join(words[:128]), {"mode": "guarded_nucleus" if sample else "guarded_greedy",
                "generated_tokens": len(generated), "forced_boundary_steps": forced,
                "closed_words": len(words), "seed": seed, "retry_token_limit": limit,
                "source_tokens_used": source.shape[1], "original_prompt_tokens": len(prompt_ids)}
    raise RuntimeError(f"Model B retry exhausted {limit} positions with {len(words)} words; last text: {text[-400:]!r}")


''')


[Markdown Cell]
Only the original word-count exhaustion triggers retries; numerical, memory, and artifact failures remain blocking errors. Record failed prompt IDs, the retry mode, token count, and constrained boundary steps in `decoding_diagnostics.jsonl`. Existing good greedy outputs remain unchanged; text comes from trained-model decoding, without copying references or appending filler.


In [17]:
# [Code Cell]
append_module('decoding.py', r'''def generate_with_recovery(model, runtime: dict, row: dict, device, token_limit: int,
                           diagnostics_path: Path) -> tuple[str, dict]:
    """Try the original inference API first; retry only its word-count exhaustion."""
    policy = runtime.get("generation_policy")
    try:
        text = runtime["infer"].generate_continuation(model, runtime["tokenizer"], row["prompt"], device, token_limit)
        return text, {"mode": "original_greedy"}
    except RuntimeError as error:
        if runtime["family"] != "b" or not policy or "Could not close" not in str(error):
            raise
        original_error = str(error)
    print(f"ID {row['id']}: greedy word-count exhaustion; retrying with repetition control", flush=True)
    errors = [original_error]
    for attempt, sample in enumerate((False, True), start=1):
        seed = (runtime["cfg"].seed + int(hashlib.sha256(row["prompt"].encode()).hexdigest()[:12], 16) + attempt) % (2**63 - 1)
        try:
            text, details = guarded_model_b(model, runtime, row["prompt"], device, token_limit, policy, sample, seed)
            details.update(id=row["id"], attempt=attempt, original_error=original_error)
            diagnostics_path.parent.mkdir(parents=True, exist_ok=True)
            with diagnostics_path.open("a", encoding="utf-8") as stream:
                stream.write(json.dumps(details, ensure_ascii=False) + "\n"); stream.flush()
            return text, details
        except RuntimeError as error:
            errors.append(str(error))
    raise RuntimeError(f"ID {row['id']}: all bounded decoding attempts failed. {errors}")
''')


[Markdown Cell]
## 11. Restart-safe generation
Use each model's original greedy cached generation API. SQLite commits each completed 128-word continuation. Namespace caches by checkpoint, notebook, official file, token limit, seed, and current Torch version; verify prompt hashes per ID. Re-running skips only matching complete predictions. Exhausting the model's context/token bound fails explicitly; no filler is added.


In [18]:
# [Code Cell]
append_module('generation.py', r'''from collections import Counter
from pathlib import Path
import hashlib
import json
import os
import shutil
import sqlite3
import zipfile
from tqdm.auto import tqdm
from .io_utils import atomic_json, read_json, sha256_file, parse_json
from .artifacts import validate_predictions
from .decoding import generate_with_recovery


def generation_identity(receipt: dict, request: dict, runtime, token_limit: int, legacy: bool = False) -> str:
    value = {"checkpoint": receipt["sha256"], "notebook": request["notebook_audit"]["notebook_sha256"],
             "test": sha256_file(Path(request["official_test_json"])), "tokens": token_limit,
             "words": 128, "torch": str(runtime["torch_version"]), "seed": runtime["cfg"].seed,
             "decoding": "original-model greedy cached decoding"}
    if runtime.get("generation_policy") and not legacy:
        value["recovery_policy"] = runtime["generation_policy"]
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def generate_records(model, runtime: dict, records: list[dict], directory: Path,
                     namespace: str, token_limit: int, device, min_free_bytes: int) -> tuple[dict, int]:
    """Commit each complete continuation; never export a partial official submission."""
    directory.mkdir(parents=True, exist_ok=True)
    connection = sqlite3.connect(directory / "prediction_cache.sqlite")
    connection.execute("PRAGMA journal_mode=WAL")
    connection.execute("PRAGMA synchronous=FULL")
    connection.execute("CREATE TABLE IF NOT EXISTS predictions (namespace TEXT,id TEXT,prompt_sha TEXT,text TEXT,PRIMARY KEY(namespace,id))")
    predictions, reused = {}, 0
    try:
        bar = tqdm(records, desc=f"Model {runtime['family'].upper()}: 128-word continuations")
        for row in bar:
            prompt_sha = hashlib.sha256(row["prompt"].encode()).hexdigest()
            cached = connection.execute("SELECT prompt_sha,text FROM predictions WHERE namespace=? AND id=?",
                                        (namespace, row["id"])).fetchone()
            if cached and cached[0] == prompt_sha and len(cached[1].split()) == 128:
                text = cached[1]; reused += 1
            else:
                if shutil.disk_usage(directory).free < min_free_bytes + (1 << 20):
                    raise OSError("Insufficient working disk space for durable predictions")
                text, details = generate_with_recovery(model, runtime, row, device, token_limit,
                    directory / "decoding_diagnostics.jsonl")
                if not isinstance(text, str) or len(text.split()) != 128:
                    raise ValueError(f"Incomplete continuation for {row['id']}")
                connection.execute("INSERT OR REPLACE INTO predictions VALUES (?,?,?,?)",
                                   (namespace, row["id"], prompt_sha, text))
                connection.commit()
            predictions[row["id"]] = {"text": text}
            bar.set_postfix(completed=len(predictions), reused=reused)
    finally:
        connection.close()
    return predictions, reused


def inherit_legacy_predictions(directory: Path, previous: Path, namespace: str,
                               previous_namespace: str, prompts: dict) -> int:
    """Reuse only validated completed greedy rows from the exact prior identity.

    Both identities include checkpoint SHA, source, official file, Torch, seed,
    and bound. Recovery changes only failed prompts; earlier good rows stay valid.
    """
    source_path = previous / "prediction_cache.sqlite"
    if not source_path.is_file():
        return 0
    directory.mkdir(parents=True, exist_ok=True)
    copied = 0
    with sqlite3.connect(source_path.resolve().as_uri() + "?mode=ro", uri=True) as source, sqlite3.connect(directory / "prediction_cache.sqlite") as target:
        target.execute("PRAGMA journal_mode=WAL")
        target.execute("PRAGMA synchronous=FULL")
        target.execute("CREATE TABLE IF NOT EXISTS predictions (namespace TEXT,id TEXT,prompt_sha TEXT,text TEXT,PRIMARY KEY(namespace,id))")
        for identifier, prompt_sha, text in source.execute(
                "SELECT id,prompt_sha,text FROM predictions WHERE namespace=?", (previous_namespace,)):
            if identifier not in prompts or prompt_sha != hashlib.sha256(prompts[identifier]["text"].encode()).hexdigest():
                continue
            try:
                validate_predictions({identifier: {"text": text}}, {identifier: prompts[identifier]})
            except (ValueError, TypeError):
                continue
            result = target.execute("INSERT OR IGNORE INTO predictions VALUES (?,?,?,?)", (namespace, identifier, prompt_sha, text))
            copied += result.rowcount
        target.commit()
    return copied


''')


[Markdown Cell]
Validate every record before export. Atomically write `prediction.json`, then a ZIP with exactly that file at the root. Check ZIP CRC and re-read JSON equality. Save `submission_verification.json`, ZIP SHA-256, samples, and repeated-continuation counts. Reports are kept outside the submitted ZIP.


In [19]:
# [Code Cell]
append_module('generation.py', r'''def export_submission(directory: Path, family: str, predictions: dict, prompts: dict) -> dict:
    result = validate_predictions(predictions, prompts)
    prediction_path = directory / "prediction.json"
    atomic_json(prediction_path, predictions)
    zip_path = directory / f"submission_model_{family}.zip"
    temporary = zip_path.with_name(zip_path.name + ".partial")
    with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(prediction_path, arcname="prediction.json")
    with temporary.open("r+b") as stream:
        os.fsync(stream.fileno())
    with zipfile.ZipFile(temporary) as archive:
        if archive.namelist() != ["prediction.json"] or archive.testzip() is not None:
            raise ValueError("Invalid ZIP root layout or CRC")
        if parse_json(archive.read("prediction.json").decode("utf-8")) != predictions:
            raise ValueError("ZIP and prediction JSON differ")
    os.replace(temporary, zip_path)
    frequencies = Counter(row["text"] for row in predictions.values())
    result.update(zip_path=str(zip_path), prediction_path=str(prediction_path),
                  zip_sha256=sha256_file(zip_path), zip_root_verified=True,
                  repeated_complete_continuation_groups=sum(count > 1 for count in frequencies.values()),
                  samples={key: predictions[key] for key in list(predictions)[:3]})
    atomic_json(directory / "submission_verification.json", result)
    return result
''')


[Markdown Cell]
## 12. Isolate each model
A fresh subprocess imports one recovered package. It uses one selected GPU and exits before the next job, releasing GPU memory and mapped checkpoint handles. It imports definitions from the training runtime without invoking its main function, optimizer creation, DDP, or training loop.


In [20]:
# [Code Cell]
append_module('worker.py', r'''from dataclasses import replace
from pathlib import Path
import argparse
import gc
import importlib
import hashlib
import json
import sys
import traceback
import torch
from tokenizers import Tokenizer
from .io_utils import atomic_json, read_json
from .artifacts import checkpoint_directories, verified_checkpoints, choose_checkpoints, official_prompts
from .checks import (load_payload, training_evidence, experiment_choices, prove_fingerprint,
                     load_trained_model, trained_forward_checks, balanced_validation_records, heldout_loss)
from .generation import generation_identity, generate_records, export_submission, inherit_legacy_predictions
from .inputs import resolve_official_test


def import_runtime(request: dict) -> dict:
    family = request["notebook_audit"]["model"]
    sys.path.insert(0, request["runtime_code_dir"])
    package = f"model_{family}"
    for name in list(sys.modules):
        if name == package or name.startswith(package + "."):
            del sys.modules[name]
    importlib.invalidate_caches()
    return {name: importlib.import_module(f"{package}.{name}")
            for name in ("config", "model", "data", "state", "train", "infer")}


def select_device(request: dict) -> torch.device:
    device = torch.device(request["device"])
    if device.type == "cuda":
        if not torch.cuda.is_available() or device.index is None or device.index >= torch.cuda.device_count():
            raise ValueError("Select a Kaggle GPU accelerator and a valid device, e.g. cuda:0")
        torch.cuda.set_device(device)
    elif device.type != "cpu":
        raise ValueError("Only CUDA or CPU inference is supported")
    if tuple(int(part) for part in str(torch.__version__).split("+")[0].split(".")[:2]) < (2, 5):
        raise ValueError("Use PyTorch >=2.5 for memory-mapped checkpoint assignment")
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    return device


''')


[Markdown Cell]
Locate/verify checkpoints, establish latest-run completion, select best weights, resolve preprocessing paths, prove the fingerprint, and build only compact document/validation indices. Verify actual trained FP32 weights before model probes. A missing/mismatched artifact becomes a precise blocking report, rather than silently falling back to fresh weights.


In [21]:
# [Code Cell]
append_module('worker.py', r'''def prepare_model(request: dict, report: dict):
    runtime = import_runtime(request)
    directory = checkpoint_directories(request)[0]
    valid, problems = verified_checkpoints(directory)
    latest, chosen = choose_checkpoints(valid)
    report["checkpoint_warnings"] = problems
    payload = load_payload(latest)
    if payload["version"] != runtime["config"].RUNTIME_VERSION:
        raise ValueError("Checkpoint version does not match the recovered architecture")
    evidence = training_evidence(payload)
    report["training"] = evidence
    report["latest_checkpoint"] = latest
    cfg = runtime["config"].TrainConfig(**payload["config"])
    cfg.validate()
    expected_choices = experiment_choices(payload["config"])
    if not evidence["complete"] and not request["allow_partial_training"]:
        raise ValueError(f"Training is incomplete: {evidence['completed_epochs']}/{evidence['planned_epochs']} epochs, next_batch={evidence['next_batch']}. Resume the original training notebook. Set allow_partial_training=True only if you intentionally want a partial-run submission.")
    if not evidence["complete"]:
        report["warnings"].append("Submission uses a deliberately allowed partial training run")
    if chosen["path"] != latest["path"]:
        del payload; gc.collect()
        payload = load_payload(chosen)
    if experiment_choices(payload["config"]) != expected_choices:
        raise ValueError("Best and latest checkpoints use different experiment configurations")
    report["selected_checkpoint"] = chosen
    requested_data = request["preprocessed_data_dir"] or cfg.data_dir
    data_dir = runtime["config"].resolve_preprocessed_data_dir(requested_data, request["input_root"])
    cfg = replace(cfg, data_dir=str(data_dir), run_dir=str(Path(request["job_dir"]) / "verified_runtime"), resume_dir=str(directory))
    if request.get("generation_token_limit") is not None:
        cfg = replace(cfg, generation_token_limit=request["generation_token_limit"])
    cfg.validate()
    contract = runtime["data"].verify_data_contract(cfg)
    original_torch = prove_fingerprint(cfg, contract, runtime, request, chosen["receipt"]["fingerprint"])
    report["compatibility"] = {"source_config_data_fingerprint": "passed", "training_torch": original_torch,
                                "inference_torch": str(torch.__version__), "data_dir": str(data_dir),
                                "tokenizer_sha256": contract["metadata"]["artifacts"]["tokenizer.json"]["sha256"]}
    if original_torch != str(torch.__version__):
        report["warnings"].append("Inference Torch differs from training; original fingerprint reproduced using saved notebook version evidence. This is inference, not a training resume.")
    # Original indexing also checks document-hash isolation and complete validation pairs.
    report["data_checks"] = runtime["data"].prepare_indices(cfg, contract)
    device = select_device(request)
    runtime["state"].seed_everything(cfg.seed, cfg.deterministic)
    model, weight_report = load_trained_model(payload, cfg, runtime, request["notebook_audit"]["model_class"], device)
    report["weights"] = weight_report
    del payload; gc.collect()
    runtime.update(cfg=cfg, tokenizer=Tokenizer.from_file(str(data_dir / "tokenizer.json")),
                   family=request["notebook_audit"]["model"], torch_version=str(torch.__version__),
                   generation_policy=request.get("generation_policy") if request["notebook_audit"]["model"] == "b" else None)
    return model, runtime, chosen, device


''')


[Markdown Cell]
Run trained-weight shape/cache/causality checks and the small balanced reference-loss sample. Optional development generation writes matching `development/prediction.json` and `truth.json`. Inspect these samples and run the supplied local evaluator for BERTScore/ROUGE-L/BLEU. Development IDs are not submitted as official predictions.


In [22]:
# [Code Cell]
append_module('worker.py', r'''def audit_model(model, runtime: dict, request: dict, report: dict, device) -> None:
    report["trained_forward_checks"] = trained_forward_checks(model, runtime["family"], runtime, device)
    if request["validation_examples_per_source"]:
        rows = balanced_validation_records(runtime["cfg"], runtime, request["validation_examples_per_source"])
        report["heldout_smoke"] = heldout_loss(model, runtime["cfg"], runtime["family"], runtime, rows, device)
        if report["heldout_smoke"]["conditional_nll"] >= report["heldout_smoke"]["uniform_vocab_nll"]:
            report["warnings"].append("Held-out smoke NLL is no better than uniform vocabulary probabilities; inspect learning and text quality")
        if request["generate_development_smoke"]:
            directory = Path(request["job_dir"]) / "development"
            namespace = report["selected_checkpoint"]["receipt"]["sha256"] + f":dev:{runtime['torch_version']}:{runtime['cfg'].generation_token_limit}"
            if runtime.get("generation_policy"):
                namespace += ":" + hashlib.sha256(json.dumps(runtime["generation_policy"], sort_keys=True).encode()).hexdigest()
            predictions, reused = generate_records(model, runtime, rows, directory, namespace,
                runtime["cfg"].generation_token_limit, device, request["min_free_bytes"])
            atomic_json(directory / "prediction.json", predictions)
            atomic_json(directory / "truth.json", {row["id"]: {"text": row["reference"]} for row in rows})
            report["development_generation"] = {"examples": len(predictions), "cached_examples": reused,
                "prediction_path": str(directory / "prediction.json"), "truth_path": str(directory / "truth.json"),
                "samples": {key: predictions[key] for key in list(predictions)[:2]}}


''')


[Markdown Cell]
## 13. Reports and generation gate
Persist phase-by-phase results and capture errors in JSON. A verified complete/compatible model can generate official predictions. An interrupted or failed job leaves durable completed predictions but no newly verified partial ZIP. Re-run to continue. `submission_ready=True` is set only after full official-ID and ZIP verification.


In [23]:
# [Code Cell]
append_module('worker.py', r'''def run_request(request_path: Path, mode: str) -> dict:
    request = read_json(request_path)
    report_path = Path(request["job_dir"]) / ("audit_report.json" if mode == "audit" else "deliverable_report.json")
    report = {"status": "checking", "mode": mode, "model": request["notebook_audit"]["model"],
              "notebook": request["notebook_audit"], "warnings": [], "submission_ready": False}
    atomic_json(report_path, report)
    try:
        test_path = None
        if mode == "generate":
            test_path = resolve_official_test(request, required=True)
        else:
            try:
                test_path = resolve_official_test(request, required=False)
            except (OSError, ValueError, TypeError, KeyError) as error:
                report["warnings"].append(f"Official test selection needs attention: {error}")
        prompts = official_prompts(test_path) if test_path else None
        report["official_test_path"] = str(test_path) if test_path else None
        report["official_test_examples"] = len(prompts) if prompts else 0
        report["official_test_ready"] = prompts is not None
        if test_path:
            request = dict(request, official_test_json=str(test_path))
        else:
            report["warnings"].append("Official test prompts are absent or unresolved; model auditing continues, but submission generation requires the test file")
        if request["notebook_audit"]["saved_error_outputs"]:
            report["warnings"].append("Notebook has saved error outputs; checkpoint and current trained checks are authoritative")
        if request["notebook_audit"].get("input_kind", "notebook") == "notebook" and not request["notebook_audit"]["executed_code_cells"]:
            report["warnings"].append("Notebook execution outputs are absent; training completion must be established from checkpoint history")
        model, runtime, chosen, device = prepare_model(request, report)
        audit_model(model, runtime, request if mode == "audit" else dict(request, generate_development_smoke=False), report, device)
        report["status"] = "ready_for_official_inference" if prompts else "model_verified"
        atomic_json(report_path, report)
        if mode == "generate":
            namespace = generation_identity(chosen["receipt"], request, runtime, runtime["cfg"].generation_token_limit)
            directory = Path(request["job_dir"]) / "official" / namespace[:16]
            if runtime.get("generation_policy"):
                previous_namespace = generation_identity(chosen["receipt"], request, runtime, runtime["cfg"].generation_token_limit, legacy=True)
                copied = inherit_legacy_predictions(directory, Path(request["job_dir"]) / "official" / previous_namespace[:16], namespace, previous_namespace, prompts)
                report["inherited_complete_greedy_predictions"] = copied
                print(f"Recovered {copied} complete predictions from the previous greedy cache", flush=True)
            rows = [{"id": key, "prompt": row["text"]} for key, row in prompts.items()]
            predictions, reused = generate_records(model, runtime, rows, directory, namespace,
                runtime["cfg"].generation_token_limit, device, request["min_free_bytes"])
            report["submission"] = export_submission(directory, runtime["family"], predictions, prompts)
            report["submission"]["cached_examples"] = reused
            report["decoding_recovery"] = {"policy": runtime.get("generation_policy"),
                "diagnostics_path": str(directory / "decoding_diagnostics.jsonl"),
                "scope": "Retries prevent token-loop word-count failures; quality requires held-out scoring"}
            diagnostics_path = directory / "decoding_diagnostics.jsonl"
            recovered = {}
            if diagnostics_path.is_file():
                with diagnostics_path.open(encoding="utf-8") as stream:
                    for line in stream:
                        try:
                            details = json.loads(line)
                            recovered[details["id"]] = details
                        except (ValueError, KeyError):
                            continue
            report["decoding_recovery"].update(retried_examples=len(recovered),
                heavily_constrained_ids=[key for key, details in recovered.items() if details.get("forced_boundary_steps", 0) > 16],
                samples={key: predictions[key] for key in list(recovered)[:2] if key in predictions})
            if any(details.get("forced_boundary_steps", 0) > 16 for details in recovered.values()):
                report["warnings"].append("Some retries needed many lexical-boundary constraints. Inspect the recovered samples: format completion does not establish semantic quality or BERTScore.")
            report.update(status="submission_verified", submission_ready=True)
            if report["submission"]["repeated_complete_continuation_groups"]:
                report["warnings"].append("Repeated full continuations across prompts: format passes, but inspect generation quality")
        del model; gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
    except (Exception, KeyboardInterrupt) as exc:
        report.update(status="blocked", error_type=type(exc).__name__, error=str(exc), submission_ready=False)
        if isinstance(exc, KeyboardInterrupt):
            report["error"] = "Interrupted; complete predictions remain cached. Re-run this job to continue."
        report["diagnostic_traceback"] = traceback.format_exc(limit=6)
    atomic_json(report_path, report)
    print(f"Model {report['model'].upper()}: {report['status']}", flush=True)
    if report.get("error"):
        print(report["error"], flush=True)
    if report["submission_ready"]:
        print(f"VERIFIED ZIP: {report['submission']['zip_path']}", flush=True)
    return report


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--request", type=Path, required=True)
    parser.add_argument("--mode", choices=("audit", "generate"), required=True)
    args = parser.parse_args()
    result = run_request(args.request, args.mode)
    raise SystemExit(0 if result["status"] != "blocked" else 1)


if __name__ == "__main__":
    main()
''')


[Markdown Cell]
## 14. Live progress and accumulated deliverables
Stream worker output and tqdm progress. Run jobs sequentially; a failure in one job is reported while other jobs remain usable. The manifest records verified ZIPs and errors across notebook reruns. Original inputs and earlier outputs are retained.


In [24]:
# [Code Cell]
append_module('client.py', r'''from pathlib import Path
import os
import signal
import subprocess
import sys
from .io_utils import read_json, atomic_json


def run_worker(request_path: Path, mode: str, hub_code_dir: Path) -> dict:
    """Run one model at a time; worker exit releases GPU and memory-mapped weights."""
    request = read_json(request_path)
    command = [sys.executable, "-u", "-m", "submission_hub.worker", "--request", str(request_path), "--mode", mode]
    environment = os.environ.copy()
    environment["TOKENIZERS_PARALLELISM"] = "false"
    environment["PYTHONUTF8"] = "1"
    process = subprocess.Popen(command, cwd=hub_code_dir, env=environment, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.terminate(); process.wait(timeout=15)
        raise
    finally:
        if process.stdout is not None:
            process.stdout.close()
    report_path = Path(request["job_dir"]) / ("audit_report.json" if mode == "audit" else "deliverable_report.json")
    report = read_json(report_path) if report_path.is_file() else {
        "model": request["notebook_audit"]["model"], "status": "blocked", "submission_ready": False,
        "error": f"Worker exited {code} without a report; inspect the output above"}
    if code and report.get("status") != "blocked":
        report.update(status="blocked", submission_ready=False, error=f"Worker exited {code} before completion; rerun to reuse completed predictions")
        atomic_json(report_path, report)
    print("Report:", report_path, flush=True)
    return report


def update_manifest(output_root: Path, reports: list[dict]) -> dict:
    path = output_root / "submission_manifest.json"
    manifest = read_json(path) if path.is_file() else {"jobs": {}}
    for report in reports:
        audit = report.get("notebook", {})
        key = f"{report.get('model', 'unknown')}:{audit.get('notebook_sha256', audit.get('notebook', 'unknown'))}"
        old = manifest["jobs"].get(key, {})
        manifest["jobs"][key] = {"model": report.get("model"), "notebook": audit.get("notebook"),
            "status": report.get("status"), "submission_ready": report.get("submission_ready", False),
            "zip_path": report.get("submission", {}).get("zip_path"),
            "previous_verified_zip": old.get("zip_path") or old.get("previous_verified_zip"),
            "error": report.get("error"), "warnings": report.get("warnings", [])}
    atomic_json(path, manifest)
    return manifest
''')


[Markdown Cell]
## 15. Materialize the standalone helper
Write only the embedded utility code to working storage. Model implementations come from the saved run's original code folder or the optional input notebook. Re-running this cell refreshes helper imports; trained runtime files are staged separately and preserve their original hashes.


In [25]:
# [Code Cell]
hub_code_dir = OUTPUT_ROOT / "hub_code"
package_dir = hub_code_dir / "submission_hub"
package_dir.mkdir(parents=True, exist_ok=True)
module_parts["__init__.py"] = ['"""CS-781 trained-notebook audit and official prediction export."""\n']
for filename, sections in module_parts.items():
    source = "".join(sections)
    compile(source, filename, "exec")
    path = package_dir / filename
    temporary = path.with_suffix(".partial")
    with temporary.open("w", encoding="utf-8", newline="\n") as stream:
        stream.write(source); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)
sys.path.insert(0, str(hub_code_dir))
for name in list(sys.modules):
    if name == "submission_hub" or name.startswith("submission_hub."):
        del sys.modules[name]
from submission_hub.notebook import stage_job
from submission_hub.client import run_worker, update_manifest
from submission_hub.io_utils import atomic_json, read_json
print("Helper runtime:", hub_code_dir)


Helper runtime: /kaggle/working/cs781_submissions/hub_code


[Markdown Cell]
## 16. Discover, inspect, and stage your selected model(s)
Read saved run code directly, or extract the optional original notebook. Review the detected model, exact run path, and saved training summary. Source/input failures are printed and persisted; checkpoint checks happen in the next cell.


In [26]:
# [Code Cell]
requests, input_errors = [], []
for job in JOBS:
    try:
        request_path = stage_job(job.notebook_path, job.run_artifacts_dir, settings, model=job.model)
        request = read_json(request_path)
        requests.append(request_path)
        print("Detected model:", request["notebook_audit"]["model"].upper())
        print("Input:", request["notebook_audit"]["notebook"], "mode:", request["notebook_audit"]["input_kind"])
        print("Saved run:", request["run_artifacts_dir"])
        print("Job folder:", request["job_dir"])
        print("Saved training summaries:", request["notebook_audit"]["training_summaries"])
    except Exception as error:
        item = {"notebook": job.notebook_path, "model": job.model, "saved_run": job.run_artifacts_dir, "status": "blocked", "error": str(error)}
        input_errors.append(item)
        print("BLOCKED:", item)
atomic_json(OUTPUT_ROOT / "input_errors.json", input_errors)
if not requests:
    print("No model was staged. Read the BLOCKED message above; input errors are saved at", OUTPUT_ROOT / "input_errors.json")


Detected model: B
Input: /kaggle/input/notebooks/shubhamjoshi1806/modelb-decoder/model_b_run mode: saved_run
Saved run: /kaggle/input/notebooks/shubhamjoshi1806/modelb-decoder/model_b_run
Job folder: /kaggle/working/cs781_submissions/model_b_739bd8cabd21
Saved training summaries: [{'status': 'complete', 'epoch': 3, 'next_batch': 0, 'step': 24267, 'optimizer_steps': 24256, 'history': [{'epoch': 1, 'train_nll': 3.5877464954897254, 'training_target_tokens': 39935991, 'overflows': 2, 'conditional_nll': 3.652189409886128, 'conditional_perplexity': 38.55899514503753, 'perplexity_capped': False, 'target_tokens': 991327, 'examples': 6279}, {'epoch': 2, 'train_nll': 2.821047039096085, 'training_target_tokens': 39936150, 'overflows': 5, 'conditional_nll': 3.336377697175039, 'conditional_perplexity': 28.117093392370275, 'perplexity_capped': False, 'target_tokens': 991327, 'examples': 6279}, {'epoch': 3, 'train_nll': 2.626181752478497, 'training_target_tokens': 39936117, 'overflows': 4, 'condition

[Markdown Cell]
## 17. Audit trained checkpoints
This loads actual weights, checks completion and compatibility, runs trained-model probes, scores the held-out smoke sample, and optionally generates development examples. A blocked model remains blocked; no random-weight fallback or automatic retraining occurs.


In [27]:
# [Code Cell]
audit_reports = []
for request_path in requests:
    report = run_worker(request_path, "audit", hub_code_dir)
    audit_reports.append(report)
    print(json.dumps({key: report.get(key) for key in
        ("model", "status", "training", "weights", "trained_forward_checks", "heldout_smoke", "warnings", "error")}, indent=2))
update_manifest(OUTPUT_ROOT, audit_reports)



Verify preprocessing receipts: 100%|██████████| 6/6 [00:03<00:00,  1.57it/s]

Model B: 128-word continuations: 100%|██████████| 8/8 [00:08<00:00,  1.09s/it, completed=8, reused=0]
Model B: ready_for_official_inference
Report: /kaggle/working/cs781_submissions/model_b_739bd8cabd21/audit_report.json
{
  "model": "b",
  "status": "ready_for_official_inference",
  "training": {
    "complete": true,
    "completed_epochs": 3,
    "planned_epochs": 3,
    "next_batch": 0,
    "saved_step": 24267,
    "successful_optimizer_updates": 24256,
    "history": [
      {
        "epoch": 1,
        "train_nll": 3.5877464954897254,
        "training_target_tokens": 39935991,
        "overflows": 2,
        "conditional_nll": 3.652189409886128,
        "conditional_perplexity": 38.55899514503753,
        "perplexity_capped": false,
        "target_tokens": 991327,
        "examples": 6279
      },
      {
        "epoch": 2,
        "train_nll": 2.821047039096085,
        "training_target_tokens": 3

{'jobs': {'b:739bd8cabd21108bbac12d1d0494826f647ac7683749c983c0fd7f5e1c47b332': {'model': 'b',
   'notebook': '/kaggle/input/notebooks/shubhamjoshi1806/modelb-decoder/model_b_run',
   'status': 'ready_for_official_inference',
   'submission_ready': False,
   'zip_path': None,
   'previous_verified_zip': None,
   'error': None,
   'warnings': ['Inference Torch differs from training; original fingerprint reproduced using saved notebook version evidence. This is inference, not a training resume.']}}}

[Markdown Cell]
## 17b. Measure development BERTScore before deciding to retrain
Use the generated held-out continuations and their matching references. This matches the supplied course evaluator: `bert_score.score(..., lang='en')`, unrescaled F1 and no IDF. Eight examples are a smoke sample, not a reliable prediction of the private leaderboard. The first run downloads the English scoring model and needs Kaggle Internet; missing dependencies/network produce a clear scoring message while submission generation remains available. Set `RUN_BERTSCORE_CHECK=False` to save this time. [BERTScore reference implementation](https://github.com/Tiiiger/bert_score).


In [28]:
# [Code Cell]
development_score_reports = []
eligible_development_jobs = []
for request_path, audit in zip(requests, audit_reports):
    if audit["status"] not in ("model_verified", "ready_for_official_inference"):
        continue
    job_dir = Path(read_json(request_path)["job_dir"])
    if (job_dir / "development/prediction.json").is_file() and (job_dir / "development/truth.json").is_file():
        eligible_development_jobs.append(job_dir)
if RUN_BERTSCORE_CHECK and eligible_development_jobs:
    import subprocess
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bert-score==0.3.13", "transformers>=4.50,<5"])
        import bert_score
        import torch
        import gc
        for job_dir in eligible_development_jobs:
            prediction_path = job_dir / "development/prediction.json"
            truth_path = job_dir / "development/truth.json"
            if not prediction_path.is_file() or not truth_path.is_file():
                print("No development pair for BERTScore; enable GENERATE_DEVELOPMENT_SMOKE and at least one example per source")
                continue
            prediction, truth = read_json(prediction_path), read_json(truth_path)
            if set(prediction) != set(truth):
                raise ValueError("Development prediction/reference IDs differ")
            ids = list(truth)
            _, _, f1 = bert_score.score([prediction[key]["text"] for key in ids],
                [truth[key]["text"] for key in ids], lang="en", batch_size=8,
                verbose=False, device=DEVICE, rescale_with_baseline=False, idf=False)
            result = {"model": audit["model"], "examples": len(ids), "bertscore_f1": float(f1.mean()),
                "scorer": "English default roberta-large; unrescaled/no IDF; course-evaluator settings",
                "per_example": {key: float(value) for key, value in zip(ids, f1)},
                "scope": "Held-out smoke sample; official private score is unknown"}
            atomic_json(job_dir / "development/bertscore_report.json", result)
            development_score_reports.append(result)
            print(json.dumps(result, indent=2))
            print("Sample:", json.dumps(prediction[ids[0]], ensure_ascii=False))
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as error:
        print("BERTScore check unavailable:", error, "— official generation remains enabled.")
elif RUN_BERTSCORE_CHECK:
    print("No verified development prediction/reference pair to score; no scoring-model download started.")
else:
    print("BERTScore check skipped for speed; score the matching development JSON later.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{
  "model": "b",
  "examples": 8,
  "bertscore_f1": 0.7864285707473755,
  "scorer": "English default roberta-large; unrescaled/no IDF; course-evaluator settings",
  "per_example": {
    "supreme_court:03709693dfd08b501547c73abcc9d75f98b93978158212c37f3debe2fc536f63:0": 0.8176546096801758,
    "supreme_court:03709693dfd08b501547c73abcc9d75f98b93978158212c37f3debe2fc536f63:1": 0.7554256916046143,
    "high_courts:f7935ec6eb36a63957fae403cbbe62b01036d9ff67bf7913805bbd5731da520f:0": 0.7768980264663696,
    "high_courts:4803d2d31ef4153aa041e8c07765b33f9a54d898caae34592183cc9e2ce394a8:0": 0.805956244468689,
    "acts:ab41d7d6a00c553e9d9681418d22c6cffcb07e2cab6a43cd949f8210f8e7461a:0": 0.7483602166175842,
    "acts:ab41d7d6a00c553e9d9681418d22c6cffcb07e2cab6a43cd949f8210f8e7461a:1": 0.7969567179679871,
    "tribunals:313aeaece625e72505d6075224a6e0b6e706752f7c403058c68c6d598e93b84f:0": 0.7852315902709961,
    "tribunals:313aeaece625e72505d6075224a6e0b6e706752f7c403058c68c6d598e93b84f:1": 0.80

[Markdown Cell]
## 18. Generate and verify official submission ZIPs
Only jobs with a successful audit proceed. Their exact official IDs are generated with trained weights and checked before ZIP publication. Set `GENERATE_OFFICIAL_SUBMISSIONS=False` for an audit-only run. Re-run after an interruption to reuse completed predictions.


In [29]:
# [Code Cell]
deliverable_reports = []
if GENERATE_OFFICIAL_SUBMISSIONS:
    for request_path, audit in zip(requests, audit_reports):
        if audit["status"] not in ("ready_for_official_inference", "model_verified"):
            print("Skipping blocked model:", audit["model"].upper(), audit.get("error"))
            continue
        if not audit.get("official_test_ready"):
            print("Model", audit["model"].upper(), "passed its audit. Attach Phase1- Test.json and rerun to generate the submission ZIP.")
            continue
        report = run_worker(request_path, "generate", hub_code_dir)
        deliverable_reports.append(report)
manifest = update_manifest(OUTPUT_ROOT, deliverable_reports)
print(json.dumps(manifest, indent=2))



Verify preprocessing receipts: 100%|██████████| 6/6 [00:01<00:00,  4.30it/s]
Recovered 0 complete predictions from the previous greedy cache

Model B: 128-word continuations:   3%|▎         | 34/1000 [00:35<16:41,  1.04s/it, completed=34, reused=0]ID 35: greedy word-count exhaustion; retrying with repetition control

Model B: 128-word continuations:   4%|▎         | 35/1000 [00:43<52:59,  3.29s/it, completed=35, reused=0]ID 36: greedy word-count exhaustion; retrying with repetition control

Model B: 128-word continuations:  10%|▉         | 97/1000 [01:53<15:37,  1.04s/it, completed=97, reused=0]ID 98: greedy word-count exhaustion; retrying with repetition control

Model B: 128-word continuations:  12%|█▎        | 125/1000 [02:28<14:50,  1.02s/it, completed=125, reused=0]ID 126: greedy word-count exhaustion; retrying with repetition control

Model B: 128-word continuations:  16%|█▌        | 162/1000 [03:12<13:40,  1.02it/s, completed=162, reused=0]ID 163: greedy word-count exhaustion; 

[Markdown Cell]
## 19. Download each verified ZIP
Upload the model's ZIP to Codabench Phase 1. Its root contains only `prediction.json`; audit reports and checkpoints stay outside. The filenames remain `submission_model_a.zip` through `submission_model_d.zip`, inside separate per-model/per-checkpoint output folders. Files also appear under Kaggle working output.


In [30]:
# [Code Cell]
for report in deliverable_reports:
    if report.get("submission_ready"):
        submission = report["submission"]
        print("MODEL", report["model"].upper(), "— VERIFIED:", submission["zip_path"])
        display(FileLink(submission["zip_path"]))
        print("Samples:", json.dumps(submission["samples"], ensure_ascii=False, indent=2))
        print("Warnings:", report["warnings"])
        print("Decoding recovery:", json.dumps(report.get("decoding_recovery", {}), ensure_ascii=False, indent=2))
    else:
        print("No verified ZIP:", report["model"].upper(), report.get("error"))
for error in input_errors:
    print("INPUT BLOCKED:", error["error"])
for audit in audit_reports:
    if audit["status"] == "blocked":
        print("MODEL", audit["model"].upper(), "BLOCKED:", audit.get("error"))
    elif not audit.get("official_test_ready"):
        print("MODEL", audit["model"].upper(), "VERIFIED — official test file required before ZIP generation.")
print("Accumulated manifest:", OUTPUT_ROOT / "submission_manifest.json")


MODEL B — VERIFIED: /kaggle/working/cs781_submissions/model_b_739bd8cabd21/official/6a59ab8f8c648af6/submission_model_b.zip


/kaggle/working/cs781_submissions/model_b_739bd8cabd21/official/6a59ab8f8c648af6/submission_model_b.zip

Samples: {
  "1": {
    "text": "the Court in the case of [State of Maharashtra & Ors. Vs. M.P. & Ors.], (2006) 4 SCC 718, the Supreme Court has held that the award cannot be said to be illegal. The Supreme Court has held that the award cannot be said to be illegal. The Supreme Court has held that the award cannot be said to be illegal. The Supreme Court has held that the award cannot be said to be illegal. The Supreme Court has held that the award cannot be said to be illegal. The Supreme Court has held that the award cannot be sustained. The Supreme Court has held that the award cannot be sustained. The Supreme Court has held that the award cannot be sustained. The writ petition is, accordingly, allowed."
  },
  "2": {
    "text": ".C. and the complaint filed by the complainant, the complainant and the complainant are not present. The complainant and the complainant are not present in the Court. The complainant and the complainant are not present in the Court. The complainant and the

[Markdown Cell]
## 20. Use the next model and interpret failures

Change `SubmissionJob(model='b')` to A, C, or D and run again, or list all four jobs.
Earlier outputs stay isolated. Download only a ZIP whose report says
`submission_ready=True`. Review sample text even when format and numerical checks
pass; repeated or irrelevant continuations may score poorly.

Missing `.pt`/receipt/code: attach saved training output. A separate `.ipynb` is
optional when the run contains its original `code/model_[a-d]` folder.
Incomplete cursor: resume the
original training notebook. Ambiguous run/data folders: set exact paths. Fingerprint
mismatch: use the original notebook, tokenizer/export, and training configuration.
Source extraction does not support unrelated notebook formats. Token/context
exhaustion: inspect the model's decoding bounds and quality; completed predictions
remain cached. The utility never fills missing words or trains a replacement model.

For held-out scoring, use the course `evaluate_phase1.py` with each job's
`development/truth.json` and `development/prediction.json`. Its BERTScore dependency
may download a scoring model; it is separate from this utility's model inference.
Official references remain private. Your capstone requires four distinct Phase-1
submissions; upload each architecture ZIP separately. Use the course competition
invitation and the live Phase-1 dates/rules.

The source is fully embedded and cell-marked. Local QA checks all four adapters,
guarded incomplete/mismatched inputs, actual parameter counts, and restart/ZIP
logic on synthetic fixtures. Your attached production checkpoints still need this
notebook's checks on Kaggle.
